## 0. Import

In [1]:
# # DOWNLOAD DATASET 
# import kagglehub

# # Thay đường dẫn thư mục ở đây
# target_dir = "/home/pc/AAA/project/deepnn/forecast-practice/dataset"

# path = kagglehub.dataset_download(
#     "youssef019/mnist-multi-digit-regression-dataset",
#     output_dir=target_dir
# )

# print("Path to dataset files:", path)

In [2]:
import numpy as np
import pandas as pd
from PIL import Image
from math import sqrt
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, Permute, Reshape, Dense,
                                     Embedding, LSTM, AdditiveAttention, Concatenate,
                                     BatchNormalization, Dropout)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

I0000 00:00:1795277395.879263  185247 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795277395.905473  185247 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795277396.458640  185247 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
IMG_H, IMG_W = 28, 224

PAD_TOKEN = 10
START_TOKEN = 11
END_TOKEN = 12

N_CLASSES = 13  # 0-9, PAD, START, END

## 1. Read dataset

In [4]:
df = pd.read_csv("/home/pc/AAA/project/deepnn/forecast-practice/datasets/labels.csv", dtype=str) # đọc dạng chuỗi để không mất số 0 ở đầu nhãn
print(df.head(), end="\n\n")
print(df.info())

   image     label
0  0.png   6290448
1  1.png    101651
2  2.png  36731246
3  3.png  24680373
4  4.png   8868017

<class 'pandas.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   image   60000 non-null  str  
 1   label   60000 non-null  str  
dtypes: str(2)
memory usage: 937.6 KB
None


In [5]:
images = []
labels = []

for image in df['image']:
    # thay đường dân folder ảnh vào đây
    image_path = "/home/pc/AAA/project/deepnn/forecast-practice/datasets/images/" + image 

    with Image.open(image_path) as img:
        img = img.convert('L')
        img = img.resize((IMG_W, IMG_H))
        img_arr = np.array(img, dtype=np.float32) / 255 # chuyển ảnh thành ma trận số và scale về [0, 1]
        images.append(img_arr)
images = np.array(images, dtype=np.float32)

for label in df['label']:
    labels.append([int(digit) for digit in label])
    
max_len = max(len(label) for label in labels)

print("Number of images:", len(images))
print("Image shape:", images.shape)
print("Maximum label length:", max_len)

Number of images: 60000
Image shape: (60000, 28, 224)
Maximum label length: 8


## 2. Preprocessing

In [6]:
# Thêm chiều kênh màu (N, 64, 128, 1) cho mạng CNN
images = np.expand_dims(np.array(images), axis=-1)

In [7]:
# Padding nhãn cho Encoder:
# Mỗi nhãn cần thêm 1 vị trí dành cho END
target_len = max_len + 1

# Khởi tạo toàn bộ bằng PAD
y = np.full((len(labels), target_len), PAD_TOKEN, dtype='int32')
for i, label in enumerate(labels):
    y[i, :len(label)] = label
    y[i, len(label)] = END_TOKEN

In [8]:
# Tạo đầu vào cho Decoder:
# Dịch nhãn sang phải 1 bước và chèn START_TOKEN vào đầu
start_col = np.full((len(y), 1), START_TOKEN, dtype='int32')
dec_in = np.concatenate([start_col, y[:, :-1]], axis=1)

In [9]:
print("Maximum digit length:", max_len)
print("Target length:", target_len)

print("\nExample labels:")
for i in range(5):
    print("Original:", df['label'].iloc[i])
    print("Decoder input:", dec_in[i])
    print("Decoder target:", y[i])
    print()

Maximum digit length: 8
Target length: 9

Example labels:
Original: 6290448
Decoder input: [11  6  2  9  0  4  4  8 12]
Decoder target: [ 6  2  9  0  4  4  8 12 10]

Original: 101651
Decoder input: [11  1  0  1  6  5  1 12 10]
Decoder target: [ 1  0  1  6  5  1 12 10 10]

Original: 36731246
Decoder input: [11  3  6  7  3  1  2  4  6]
Decoder target: [ 3  6  7  3  1  2  4  6 12]

Original: 24680373
Decoder input: [11  2  4  6  8  0  3  7  3]
Decoder target: [ 2  4  6  8  0  3  7  3 12]

Original: 8868017
Decoder input: [11  8  8  6  8  0  1  7 12]
Decoder target: [ 8  8  6  8  0  1  7 12 10]



## 3. Data Split

Chia train/val/test = 70/15/15

In [10]:
idx = np.arange(len(images))
idx_train, idx_tmp = train_test_split(idx, test_size=0.30, random_state=42)
idx_val, idx_test = train_test_split(idx_tmp, test_size=0.50, random_state=42)

X_train, X_val, X_test = images[idx_train], images[idx_val], images[idx_test]
D_train, D_val, D_test = dec_in[idx_train], dec_in[idx_val], dec_in[idx_test]
y_train, y_val, y_test = y[idx_train], y[idx_val], y[idx_test]

print('Train:', X_train.shape)
print('Validation:', X_val.shape)
print('Test:', X_test.shape)

Train: (42000, 28, 224, 1)
Validation: (9000, 28, 224, 1)
Test: (9000, 28, 224, 1)


## 4. Train model

In [ ]:
units = 256

# Encoder (CNN + LSTM)
img_input = Input(shape=(IMG_H, IMG_W, 1))          # (28, 224, 1)
x = Conv2D(32, (3, 3), padding='same', activation='relu')(img_input)    # (28, 224, 32)
x = BatchNormalization()(x)
x = MaxPooling2D((2, 2))(x)                         # (14, 112, 32)
x = Conv2D(64, (3, 3), padding='same', activation='relu')(x)            # (14, 112, 64)
x = BatchNormalization()(x)
x = MaxPooling2D((2, 2))(x)                         # (7, 56, 64)
x = Conv2D(128, (3, 3), padding='same', activation='relu')(x)           # (7, 56, 128)
x = BatchNormalization()(x)
x = MaxPooling2D((2, 2))(x)                         # (3, 28, 128)

# Chuyển feature map thành chuỗi theo chiều ngang
# feature map (H', W', C) -> chuỗi W' vector, mỗi vector mô tả một "cột" của ảnh
h, w, c = x.shape[1], x.shape[2], x.shape[3]
x = Permute((2, 1, 3))(x)
x = Reshape((w, h * c))(x)                          # (28, 3*128) = (28, 384)  

enc_out, state_h, state_c = LSTM(units, return_sequences=True, return_state=True)(x)

# Decoder (LSTM + Additive Attention)
dec_input = Input(shape=(target_len,))
emb = Embedding(N_CLASSES, 64)(dec_input)
dec_out = LSTM(units, return_sequences=True)(emb, initial_state=[state_h, state_c])

context = AdditiveAttention()([dec_out, enc_out])
merged = Concatenate()([dec_out, context])
z = Dense(256, activation='relu')(merged)
outputs = Dense(N_CLASSES, activation='softmax')(z)

I0000 00:00:1795277402.522876  185247 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 11951 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Ti SUPER, pci bus id: 0000:01:00.0, compute capability: 8.9


In [12]:
model = Model(inputs=[img_input, dec_input], outputs=outputs)
model.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', weighted_metrics=['sparse_categorical_accuracy'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 28, 224,   │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 28, 224,   │        320 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 28, 224,   │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 14, 112,   │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 14, 112,   │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 14, 112,   │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 7, 56, 64) │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 7, 56,     │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 7, 56,     │        512 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 3, 28,     │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ permute (Permute)   │ (None, 28, 3,     │          0 │ max_pooling2d_2[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 9)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 28, 384)   │          0 │ permute[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 9, 64)     │        832 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 28, 256), │    656,384 │ reshape[0][0]     │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 9, 256)    │    328,704 │ embedding[0][0],  │
│                     │                   │            │ lstm[0][1],       │
│                     │                   │            │ lstm[0][2]        │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 1,214,413 (4.63 MB)

 Trainable params: 1,213,965 (4.63 MB)

 Non-trainable params: 448 (1.75 KB)

In [13]:
# Mask những vị trí PAD
train_mask = (y_train != PAD_TOKEN).astype('float32')
val_mask = (y_val != PAD_TOKEN).astype('float32')

# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)

# Training
history = model.fit([X_train, D_train], y_train, 
                    sample_weight=train_mask, 
                    epochs=50, 
                    batch_size=64, 
                    verbose=1, 
                    validation_data=([X_val, D_val], y_val, val_mask), 
                    callbacks=[early_stop, reduce_lr])

Epoch 1/50


I0000 00:00:1795277404.864705  185457 cuda_dnn.cc:461] Loaded cuDNN version 92801


657/657 ━━━━━━━━━━━━━━━━━━━━ 19s 25ms/step - loss: 0.4086 - sparse_categorical_accuracy: 0.8149 - val_loss: 0.1037 - val_sparse_categorical_accuracy: 0.9581 - learning_rate: 0.0010
Epoch 2/50
657/657 ━━━━━━━━━━━━━━━━━━━━ 16s 24ms/step - loss: 0.0271 - sparse_categorical_accuracy: 0.9901 - val_loss: 0.0354 - val_sparse_categorical_accuracy: 0.9862 - learning_rate: 0.0010
Epoch 3/50
657/657 ━━━━━━━━━━━━━━━━━━━━ 16s 24ms/step - loss: 0.0130 - sparse_categorical_accuracy: 0.9951 - val_loss: 0.0205 - val_sparse_categorical_accuracy: 0.9926 - learning_rate: 0.0010
Epoch 4/50
657/657 ━━━━━━━━━━━━━━━━━━━━ 16s 24ms/step - loss: 0.0105 - sparse_categorical_accuracy: 0.9960 - val_loss: 0.0122 - val_sparse_categorical_accuracy: 0.9956 - learning_rate: 0.0010
Epoch 5/50
657/657 ━━━━━━━━━━━━━━━━━━━━ 16s 25ms/step - loss: 0.0109 - sparse_categorical_accuracy: 0.9960 - val_loss: 0.0115 - val_sparse_categorical_accuracy: 0.9955 - learning_rate: 0.0010
Epoch 6/50
657/657 ━━━━━━━━━━━━━━━━━━━━ 16s 25ms/st

## 5. Prediction

In [14]:
# khi dự đoán không có nhãn thật nên phải sinh từng chữ số một,
# chữ số vừa sinh ra được đưa vào làm input cho bước kế tiếp

# Khởi tạo Decoder input
d = np.full((len(X_test), target_len), PAD_TOKEN, dtype='int32')
d[:, 0] = START_TOKEN

# Lưu kết quả dự đoán
y_pred = np.full((len(X_test), target_len), PAD_TOKEN, dtype='int32')
finished = np.zeros(len(X_test), dtype=bool)

# Greedy Decoding
for t in range(target_len):
    probs = model.predict([X_test, d], batch_size=256, verbose=0)
    next_token = np.argmax(probs[:, t, :], axis=-1)

    # Những chuỗi đã kết thúc không dự đoán thêm
    next_token[finished] = PAD_TOKEN
    y_pred[:, t] = next_token

    # Kiểm tra chuỗi đã gặp END chưa
    finished = finished | (next_token == END_TOKEN)
    
    # Đưa token vừa dự đoán vào bước tiếp theo
    if t + 1 < target_len:
        active = ~finished
        d[active, t + 1] = next_token[active]

## 6. Evaluation

In [15]:
# Chuyển chuỗi token thành chuỗi chữ số
def decode_sequence(row):
    digits = []
    for token in row:
        if token == END_TOKEN or token == PAD_TOKEN:
            break
        if 0 <= token <= 9:
            digits.append(str(token))
    return ''.join(digits)


# Chuỗi thực tế
true_sequences = [
    ''.join(str(d) for d in labels[i])
    for i in idx_test
]

# Chuỗi dự đoán
pred_sequences = [
    decode_sequence(row)
    for row in y_pred
]

In [16]:

# SEQUENCE ACCURACY
correct = np.array([true == pred for true, pred in zip(true_sequences, pred_sequences)])
seq_acc = np.mean(correct)
print(f"Sequence Accuracy: {seq_acc:.4f}")

Sequence Accuracy: 0.9899


In [ ]:
# NUMERIC METRICS
# Chỉ tính trên các mẫu mà cả actual và prediction đều là chuỗi hợp lệ
valid = np.array([
    true != '' and pred != ''
    for true, pred in zip(true_sequences, pred_sequences)
])

true_num = np.array([
    int(true) for true in np.array(true_sequences)[valid]
], dtype=np.float64)

pred_num = np.array([
    int(pred) for pred in np.array(pred_sequences)[valid]
], dtype=np.float64)

mse = mean_squared_error(true_num, pred_num)
mae = mean_absolute_error(true_num, pred_num)
r2 = r2_score(true_num, pred_num)
# NSE
nse_denominator = np.sum((true_num - np.mean(true_num)) ** 2)
nse = 1 - np.sum((true_num - pred_num) ** 2) / nse_denominator if nse_denominator != 0 else np.nan

print(f"MAE:  {mae:.4f}")
print(f"RMSE: {sqrt(mse):.4f}")
print(f"R2:   {r2:.4f}")
print(f"NSE:  {nse:.4f}")


MAE:  7490.5826
RMSE: 442832.3042
R2:   0.9995
NSE:  0.9995
